In [2]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.0 MB/s eta 0:00:00


In [3]:
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
import time

# REQUIRED
Entrez.email = "noreply.studyhubteam@gmail.com"


def search_pubmed(query, max_results=2000):
    """Search PubMed using history server to support large result sets."""
    handle = Entrez.esearch(
        db="pubmed",
        term=query,
        retmax=max_results,
        usehistory="y"       # Required for reliable large fetches
    )
    results = Entrez.read(handle)
    handle.close()
    return results["IdList"]


def fetch_abstracts(id_list, batch_size=200):
    records = []

    for start in tqdm(range(0, len(id_list), batch_size)):
        end = min(start + batch_size, len(id_list))
        batch_ids = id_list[start:end]

        try:
            handle = Entrez.efetch(
                db="pubmed",
                id=batch_ids,
                rettype="abstract",
                retmode="xml"
            )
            results = Entrez.read(handle)
            handle.close()

            for article in results["PubmedArticle"]:
                try:
                    pmid = str(article["MedlineCitation"]["PMID"])
                    article_data = article["MedlineCitation"]["Article"]

                    # Extract Abstract
                    if "Abstract" in article_data:
                        abstract_parts = article_data["Abstract"]["AbstractText"]
                        # Handle structured abstracts (list) vs plain string
                        if isinstance(abstract_parts, list):
                            abstract = " ".join(str(p) for p in abstract_parts)
                        else:
                            abstract = str(abstract_parts)
                    else:
                        abstract = ""

                    title = str(article_data.get("ArticleTitle", ""))

                    records.append({
                        "PMID": pmid,
                        "Title": title,
                        "Abstract": abstract
                    })

                except Exception:
                    continue

            time.sleep(0.4)  # Stay within NCBI rate limits (3 req/sec)

        except Exception as e:
            print(f"  Batch error (start={start}): {e}")
            time.sleep(3)

    return records


# ─── Queries ────────────────────────────────────────────────────────────────
# 8 queries × 2000 = ~16k fetched → well above 12k after dedup + empty filter
queries = [
    "diabetes mellitus",
    "cancer neoplasm",
    "cardiovascular disease",
    "neurological disorders",
    "stroke cerebrovascular",
    "hypertension blood pressure",
    "Alzheimer disease dementia",
    "Parkinson disease"
]

TARGET_PER_QUERY = 2000   # 8 × 2000 = 16,000 raw → ~12k+ after cleaning

# ─── Collection loop ─────────────────────────────────────────────────────────
all_data = []
seen_pmids = set()   # Track globally to avoid cross-query duplicates

for q in queries:
    print(f"\n{'='*60}")
    print(f"Collecting for: {q}")
    print(f"{'='*60}")

    ids = search_pubmed(q, max_results=TARGET_PER_QUERY)
    print(f"  Found {len(ids)} IDs")

    # Filter already-seen PMIDs before fetching (saves API calls)
    new_ids = [i for i in ids if i not in seen_pmids]
    print(f"  New (unseen) IDs: {len(new_ids)}")

    if not new_ids:
        print("  Skipping — all duplicates.")
        continue

    data = fetch_abstracts(new_ids)

    for item in data:
        item["Disease"] = q
        seen_pmids.add(item["PMID"])

    all_data.extend(data)

    # Save incremental progress after each query
    pd.DataFrame(all_data).to_csv("pubmed_progress.csv", index=False)
    print(f"  Collected {len(data)} records | Total so far: {len(all_data)}")

    # Stop early if we already have plenty
    if len(all_data) >= 20000:
        print("\nReached 20k records — stopping early.")
        break

# ─── Final cleaning ───────────────────────────────────────────────────────────
df = pd.DataFrame(all_data)

# Remove empty abstracts
df = df[df["Abstract"].str.strip() != ""]

# Remove duplicate PMIDs (keep first occurrence = first disease label)
df = df.drop_duplicates(subset=["PMID"])

# Optional: also deduplicate on abstract text
df = df.drop_duplicates(subset=["Abstract"])

# Reset index
df = df.reset_index(drop=True)

# Save final dataset  ← fixed: added .csv extension
df.to_csv("pubmed_dataset.csv", index=False)

print(f"\n{'='*60}")
print(f"Final dataset size: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Disease distribution:\n{df['Disease'].value_counts()}")
print("Saved to pubmed_dataset.csv")


  Found 2000 IDs
  New (unseen) IDs: 2000


100%|██████████| 10/10 [00:24<00:00,  2.44s/it]


  Collected 1999 records | Total so far: 1999

  Found 2000 IDs
  New (unseen) IDs: 1982


100%|██████████| 10/10 [00:25<00:00,  2.53s/it]


  Collected 1982 records | Total so far: 3981

  Found 2000 IDs
  New (unseen) IDs: 1794


...

  Collected 1845 records | Total so far: 11160

  Found 2000 IDs
  New (unseen) IDs: 1808


100%|██████████| 10/10 [00:26<00:00,  2.67s/it]


  Collected 1806 records | Total so far: 12966

  Found 2000 IDs
  New (unseen) IDs: 1688


100%|██████████| 9/9 [00:21<00:00,  2.42s/it]


  Collected 1687 records | Total so far: 14653

Final dataset size: 14065
Columns: ['PMID', 'Title', 'Abstract', 'Disease']
Disease distribution:
Disease
cancer neoplasm                1951
diabetes mellitus              1922
stroke cerebrovascular         1880
Alzheimer disease dementia     1753
hypertension blood pressure    1747
cardiovascular disease         1722
neurological disorders         1586
Parkinson disease              1504
Name: count, dtype: int64
Saved to pubmed_dataset.csv
